# NB04_Satellite_Musanga_Final — multiscale Musanga fraction and BRDF sensitivity

**New notebook; the validated earlier NB04 is untouched.** Five target resolutions: **10, 30, 50, 75, 100 m**. Reuse per-resolution cached UAV fractions and satellite features where available. Use the same **forest-balanced, abundance-stratified, geographically buffered train/calibration/test** approach as verified NB04 (Sections 06–07), including **three held-out repeats**, unequal site sample sizes, calibration and uncertainty metrics.

### Safety / interpretation
- By default this notebook is **audit-only** (`DRY_RUN=True`), avoiding accidental lengthy fitting or expensive re-extraction.
- **No new satellite prediction maps** are produced; this is an experiment notebook reporting held-out model comparisons. The four published 100 m GeoTIFFs remain authoritative.
- Original per-scale parquet tables are reused without recomputing UAV segmentation; absent scales must be generated from full-resolution UAV probability mosaic and satellite predictors on the same lattice **before** that scenario is run. Do not silently upscale/downscale model predictions.
- **BRDF corrected input is not yet verified.** Exact original V3 processing needs recovery and validation against existing V3 corrected data. BRDF scenarios are **never** run on RAW renamed as BRDF.
- Comparisons between RAW and BRDF for a given scale use matching row/col holdouts, same UAV targets, ensemble and training protocol. Between-scale comparisons use the same physical block size and geographic buffering; because cell supports differ, they are not necessarily identical individual observations.
- Important: 10 m resolution may be more autocorrelated and more demanding. Interpret cross-scale performance carefully; record training sample support and footprint coverage.

In [ ]:
# 01 — Drive, paths and scenario discovery (safe, no exports/retraining)
%pip -q install rasterio pandas pyarrow numpy scipy matplotlib scikit-learn
from pathlib import Path
import pandas as pd, numpy as np, json
from IPython.display import display
from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir(): drive.mount("/content/drive")
ROOT=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
CACHE=ROOT/"Final/Outputs/V3_1/04B_roi_quarterly_LOSO/samples"
FINAL=ROOT/"Final/Outputs/V3_1/NB04_Satellite_Musanga_Final"
FINAL.mkdir(parents=True,exist_ok=True)
SCALES=[10,30,50,75,100]
SITES=["Baego","MOS","Yangambi","Yoko"]
RUN_RAW=True
RUN_BRDF=True  # only if verified corrected predictor parquet exists
FORCE_VALIDATION=False
DRY_RUN=True   # set False after reviewing audit; no training by default
# BRDF corrected tables: exact same site/group/row/col, UAV target, embeddings and S2 schema.
# Do not confuse this with old (raw) roi_cells_* tables.
BRDF_TEMPLATE="roi_cells_{scale}m_BRDF.parquet"
audit=[]
for scale in SCALES:
    raw=CACHE/f"roi_cells_{scale}m.parquet"
    corrected=CACHE/BRDF_TEMPLATE.format(scale=scale)
    for kind,p in [("RAW",raw),("BRDF",corrected)]:
        rec=dict(scale_m=scale,variant=kind,path=str(p),exists=p.exists(),n_cells=0,features=0)
        if p.exists():
            try:
                z=pd.read_parquet(p)
                rec["n_cells"]=len(z)
                rec["features"]=sum(c.startswith(("EMB_ANNUAL__","S2_QUARTERLY_3YR__")) for c in z.columns)
                rec["sites"]=",".join(sorted(map(str,z.site.unique()))) if "site" in z else "MISSING"
                rec["valid"]=bool({"site","group","row","col","mean_probability"}.issubset(z.columns)
                                  and rec["features"]>=104 and set(z.site.unique())==set(SITES))
            except Exception as e:rec["error"]=repr(e)
        audit.append(rec)
inventory=pd.DataFrame(audit)
display(inventory)
inventory.to_csv(FINAL/"input_inventory.csv",index=False)
print("RAW parquet caches will be reused: no UAV resegmentation or satellite exports.")
print("BRDF comparison requires distinct, genuinely BRDF-corrected tables, NOT renamed RAW files.")
print("Set DRY_RUN=False to fit all available scenarios.")


## 01B — Find existing V3 BRDF rasters before new extraction

The original V3 Drive directory contains BRDF collections named `S2_BRDF_T3Y` and `S2_BRDF_QUARTERLY`. **Reuse these if — and only if — their year windows, preprocessing, band order and pixel grid can be matched to the raw sample caches.** This audit recursively lists candidate files without downloading anything or changing datasets. If more than one candidate exists for a site, select the correct file explicitly rather than accepting an ambiguous match. Recompute missing UAV fractions only where unavoidable, always anchored to the existing reference grid.

The BRDF experiment expects one 44-band quarterly stack per site: 10 spectral reflectance bands plus a clear-observation count in each of four quarters. If existing V3 BRDF rasters differ in feature order or temporal window, an adaptation step is mandatory before modelling.

In [ ]:
# 01B — Read-only V3 BRDF source discovery (no downloads)
from pathlib import Path
V3_FEATURES=ROOT/"Final/Outputs/V3/04_satellite_Musanga/features"
DISCOVER_TERMS=("BRDF","brdf")
print("V3 feature folder:",V3_FEATURES,"exists:",V3_FEATURES.is_dir())
candidates=[]
if V3_FEATURES.is_dir():
    for p in V3_FEATURES.rglob("*"):
        if p.is_file() and p.suffix.lower() in (".tif",".tiff",".vrt",".parquet"):
            if any(t in str(p) for t in DISCOVER_TERMS):
                candidates.append(p)
for p in sorted(candidates):
    print("V3 BRDF SOURCE:",str(p.relative_to(ROOT)),flush=True)
if not candidates:
    print("No BRDF files found in the expected V3 features directory.")
    print("Look for folders S2_BRDF_T3Y and S2_BRDF_QUARTERLY on Drive.")
    print("Do not run BRDF comparisons until these inputs have been located.")


## 02 — Frozen established modeling core

The following two cells store copies of Sections **06** and **07** from the verified NB04. They are not automatically executed. The only substitutions are (i) externally defined scale, (ii) scenario input path, and (iii) scenario-specific validation directory. This avoids divergent model logic while keeping every result isolated. The code can be inspected here to audit scientific reproducibility.

In [ ]:
BASELINE_06 = "# 06 — INPUTS, settings, forest-balanced sampling & spatial partitions\n# Run ONLY sections 06 onward. No raster extraction or legacy setup needed.\nfrom pathlib import Path\nimport numpy as np, pandas as pd, rasterio, matplotlib.pyplot as plt, json\nfrom scipy.spatial import cKDTree\nfrom sklearn.ensemble import HistGradientBoostingRegressor\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.decomposition import PCA\nfrom sklearn.neighbors import NearestNeighbors\nfrom sklearn.metrics import r2_score\nfrom IPython.display import display\nfrom google.colab import drive\nif not Path(\"/content/drive/MyDrive\").is_dir():drive.mount(\"/content/drive\")\nROOT=Path(\"/content/drive/MyDrive/Colab Notebooks/Musanga\")\nCACHE=ROOT/\"Final/Outputs/V3_1/04B_roi_quarterly_LOSO/samples\"\nSAT=ROOT/\"Final/Outputs/V3/04_satellite_Musanga/features/RAW_ANNUAL\"\nOUT=ROOT/\"Final/Outputs/V3_1/04_uncertainty_mapping\"\nOUT.mkdir(parents=True,exist_ok=True)\n# Settings: change here; validation and map stages use the same values.\nSCALE_M=EXPERIMENT_SCALE_M  # externally controlled; do not change in this cell\nYEARS={\"Baego\":2023,\"MOS\":2024,\"Yangambi\":2024,\"Yoko\":2020}\nFORESTS=[\"Baego\",\"MOS\",\"Yangambi\",\"Yoko\"]\nFEATURE_FAMILIES=[\"EMB_ANNUAL\",\"S2_QUARTERLY_3YR\"]\nN_ENSEMBLE=5\nTRAIN_CAP_PER_FOREST=500  # equal forest contribution; MOS cannot dominate by size\nTRAIN_STRATEGY='stratified'  # baseline='random'; stratify ONLY within training partition\nABUNDANCE_EDGES=[0,.10,.20,.40,1.000001]\nMIN_HIGH_SHARE=.25  # target high >=40% where available; no synthetic oversampling\nBLOCK_SIZE_M=500\nSPATIAL_BUFFER_M=150\nTEST_FRACTION=.20\nCAL_FRACTION=.20\nINTERVAL_LEVEL=.90\nREPEATS=3\nSEED=20261022  # same seed for BRDF vs raw within a resolution\nFORCE_VALIDATION=False  # existing scenario-specific heldout outputs reused\n# Required artifact, aligned original Block 07 satellite/UAV sampling\nSAMPLE_PATH=EXPERIMENT_SAMPLE_PATH  # independently audited baseline or BRDF cache\nassert SAMPLE_PATH.is_file(),f\"Cached data missing: {SAMPLE_PATH}\"\nall_cells=pd.read_parquet(SAMPLE_PATH).reset_index(drop=True)\nassert set(all_cells.site.unique())==set(FORESTS),all_cells.site.unique()\nassert {\"site\",\"group\",\"mosaic\",\"row\",\"col\",\"mean_probability\"}.issubset(all_cells.columns)\nFEATURE_COLS=[c for c in all_cells if any(c.startswith(f+\"__\") for f in FEATURE_FAMILIES)]\nassert len(FEATURE_COLS)>60,\"Missing embedding/seasonal feature columns\"\nassert all_cells[FEATURE_COLS].notna().all().all()\nassert all_cells.mean_probability.between(0,1).all()\nassert not all_cells.duplicated([\"site\",\"group\",\"row\",\"col\"]).any(),\"Duplicate spatial samples\"\n# A cell's row/col are its position on that forest's satellite grid (not geodetic coordinates).\n# Group-prefixed spatial blocks prevent falsely merging the two MOS mosaics.\ndf=all_cells.copy()\ndf[\"x_m\"]=(df.col.to_numpy(float)+.5)*SCALE_M\ndf[\"y_m\"]=(df.row.to_numpy(float)+.5)*SCALE_M\ndf[\"block_id\"]=(df.group.astype(str)+\"_\"+\n    np.floor(df.x_m/BLOCK_SIZE_M).astype(int).astype(str)+\"_\"+\n    np.floor(df.y_m/BLOCK_SIZE_M).astype(int).astype(str))\nprint(\"CACHED SPATIAL SUPPORT:\",len(df),\"cells at\",SCALE_M,\"m\")\ndisplay(df.groupby(\"site\").mean_probability.agg(\n    n=\"size\",mean_cover=\"mean\",p90=lambda s:s.quantile(.9),\n    high_fraction=lambda s:(s>=.4).mean()).round(3))\nprint(\"Full-region feature rasters beyond the UAV footprints are NOT available in this cache.\")\ndef subset_cap(frame,max_n,rng):\n    if len(frame)<=max_n:return frame.copy()\n    return frame.iloc[np.sort(rng.choice(len(frame),max_n,replace=False))].copy()\ndef stratified_cap(frame,max_n,rng):\n    # Oversample the *selection share* of high covers, never duplicate cells;\n    # all sampling happens after geographic train/cal/test splitting.\n    if len(frame)<=max_n:return frame.copy()\n    x=frame.copy()\n    x[\"_abundance_bin\"]=pd.cut(x.mean_probability,ABUNDANCE_EDGES,\n                               labels=False,include_lowest=True,right=False)\n    pool={int(k):group.index.to_numpy() for k,group in x.groupby(\"_abundance_bin\")}\n    chosen=[];high=pool.get(3,np.array([],dtype=int))\n    n_high=min(len(high),int(np.ceil(max_n*MIN_HIGH_SHARE)))\n    if n_high:chosen.extend(rng.choice(high,size=n_high,replace=False).tolist())\n    remaining=max_n-len(chosen)\n    # Quotas for remaining bins are distributed evenly among available classes.\n    leftovers={k:np.setdiff1d(ids,np.array(chosen,dtype=int)) for k,ids in pool.items()}\n    while remaining>0:\n        active=[k for k,ids in leftovers.items() if len(ids)>0]\n        if not active:break\n        n_each=max(1,remaining//len(active))\n        progress=0\n        for k in active:\n            m=min(n_each,len(leftovers[k]),remaining)\n            if not m:continue\n            picked=rng.choice(leftovers[k],size=m,replace=False)\n            chosen.extend(picked.tolist())\n            leftovers[k]=np.setdiff1d(leftovers[k],picked)\n            remaining-=m;progress+=m\n            if remaining==0:break\n        if progress==0:break\n    assert len(chosen)==max_n,(len(chosen),max_n)\n    return frame.loc[chosen].copy()\ndef balanced_training(frame,rng):\n    portions=[]\n    for site in FORESTS:\n        part=frame[frame.site==site]\n        if TRAIN_STRATEGY==\"stratified\":\n            d=stratified_cap(part,TRAIN_CAP_PER_FOREST,rng)\n        elif TRAIN_STRATEGY==\"random\":\n            d=subset_cap(part,TRAIN_CAP_PER_FOREST,rng)\n        else:raise ValueError(TRAIN_STRATEGY)\n        if len(d)<30:raise ValueError(f\"Too few training rows for {site}: {len(d)}\")\n        portions.append(d)\n    return pd.concat(portions,ignore_index=True)\ndef split_spatial(frame,repeat):\n    rng=np.random.default_rng(SEED+SCALE_M*100+repeat)\n    tr_parts=[];ca_parts=[];te_parts=[]\n    for site in FORESTS:\n        whole=frame[frame.site==site].copy()\n        groups=whole.groupby(\"block_id\").indices\n        keys=list(groups);rng.shuffle(keys)\n        held={\"test\":[],\"cal\":[]};cursor=0\n        for split,fraction in [(\"test\",TEST_FRACTION),(\"cal\",CAL_FRACTION)]:\n            count=0\n            while cursor<len(keys) and count<fraction*len(whole):\n                key=keys[cursor];cursor+=1\n                held[split].append(key);count+=len(groups[key])\n        a=whole[~whole.block_id.isin(held[\"test\"]+held[\"cal\"])].copy()\n        b=whole[whole.block_id.isin(held[\"cal\"])].copy()\n        c=whole[whole.block_id.isin(held[\"test\"])].copy()\n        # Real-world distances only within the same forest footprint.\n        def remove_near(candidate,reference):\n            if candidate.empty or reference.empty:return candidate.iloc[0:0].copy()\n            nearest=cKDTree(reference[[\"x_m\",\"y_m\"]].to_numpy()).query(\n                candidate[[\"x_m\",\"y_m\"]].to_numpy(),k=1)[0]\n            return candidate.iloc[np.flatnonzero(nearest>=SPATIAL_BUFFER_M)].copy()\n        b=remove_near(b,a);c=remove_near(c,a);c=remove_near(c,b)\n        if min(len(a),len(b),len(c))<8:\n            raise ValueError((\"Insufficient independent blocks\",site,repeat,\n                              len(a),len(b),len(c),\"Try smaller buffer or larger coverage\"))\n        tr_parts.append(a);ca_parts.append(b);te_parts.append(c)\n    return tuple(pd.concat(items,ignore_index=True) for items in (tr_parts,ca_parts,te_parts))\ndef fit_ensemble(train,predict_sets,seed):\n    models=[];rng=np.random.default_rng(seed)\n    for i in range(N_ENSEMBLE):\n        fit=balanced_training(train,np.random.default_rng(seed+i*113))\n        # Ensemble varies training draws within each forest; fixed boosting settings.\n        model=HistGradientBoostingRegressor(learning_rate=.06,max_iter=200,\n            max_leaf_nodes=20,min_samples_leaf=40,l2_regularization=2,\n            early_stopping=False,random_state=seed+i)\n        model.fit(fit[FEATURE_COLS].to_numpy(np.float32),\n                  fit.mean_probability.to_numpy(np.float32),\n                  sample_weight=fit.site.map(fit.site.value_counts().rdiv(1)).to_numpy(float)\n                    / np.mean(fit.site.map(fit.site.value_counts().rdiv(1)).to_numpy(float)))\n        models.append(model)\n    outputs=[]\n    for group in predict_sets:\n        x=group[FEATURE_COLS].to_numpy(np.float32)\n        arr=np.stack([np.clip(m.predict(x),0,1) for m in models],axis=0)\n        outputs.append((arr.mean(axis=0),arr.std(axis=0)))\n    return models,outputs\n"
print('Loaded verified 06 code:',len(BASELINE_06.splitlines()),'lines')

In [ ]:
BASELINE_07 = "# 07 — SPATIALLY RESERVED TRAIN / CALIBRATION / TEST & global, equal-forest metrics\nVALID_DIR=EXPERIMENT_VALID_DIR;VALID_DIR.mkdir(parents=True,exist_ok=True)\ndef empirical_quantile(values,level):\n    v=np.sort(np.asarray(values,float))\n    k=min(len(v),int(np.ceil((len(v)+1)*level)))\n    return float(v[k-1])\ndef forest_balanced_calibration(cal,pred):\n    # Calibrate *signed* residual tails. Equal-forest subsample prevents MOS\n    # setting the error distribution; predicted bins make interval shape adaptive.\n    # These are empirical, not coverage-guaranteed under spatial/domain shift.\n    rng=np.random.default_rng(SEED+824)\n    n_each=int(cal.site.value_counts().min())\n    idx=np.concatenate([rng.choice(np.flatnonzero(cal.site.to_numpy()==forest),\n                       n_each,replace=False) for forest in FORESTS])\n    error=cal.mean_probability.to_numpy(float)[idx]-np.asarray(pred,float)[idx]\n    fitted=np.asarray(pred,float)[idx]\n    alpha=1-INTERVAL_LEVEL\n    def signed_limits(v):\n        v=np.sort(np.asarray(v,float));n=len(v)\n        # Conservative signed-residual empirical quantiles.\n        lo=v[max(0,int(np.floor((n+1)*alpha/2))-1)]\n        hi=v[min(n-1,int(np.ceil((n+1)*(1-alpha/2)))-1)]\n        return [float(lo),float(hi)]\n    global_limits=signed_limits(error)\n    # Bins depend ONLY on predicted fractions; sparse strata use pooled limits.\n    edges=[0,.10,.25,1.000001]\n    bins=[]\n    for a,b in zip(edges[:-1],edges[1:]):\n        within=(fitted>=a)&(fitted<b)\n        n=int(within.sum())\n        limits=signed_limits(error[within]) if n>=25 else global_limits\n        bins.append(dict(min_pred=a,max_pred=b,n_cal=n,limits=limits,\n                         pooled_fallback=n<25))\n    return dict(n_per_forest=n_each,n_total=len(idx),global_limits=global_limits,\n                bins=bins,bin_edges=edges)\ndef interval_bounds(pred,calibration):\n    pred=np.asarray(pred,float)\n    low=np.empty_like(pred);high=np.empty_like(pred)\n    for segment in calibration[\"bins\"]:\n        mask=(pred>=segment[\"min_pred\"])&(pred<segment[\"max_pred\"])\n        lo,hi=segment[\"limits\"]\n        low[mask]=np.clip(pred[mask]+lo,0,1)\n        high[mask]=np.clip(pred[mask]+hi,0,1)\n    assert np.all(high>=low),\"Asymmetric interval boundaries invalid\"\n    return low,high\ndef score(sub):\n    y=sub.observed.to_numpy(float);p=sub.predicted.to_numpy(float)\n    high=y>=.4\n    return dict(n=len(sub),R2=r2_score(y,p) if len(y)>1 and y.std()>1e-8 else np.nan,\n                MAE_pp=100*np.mean(abs(y-p)),\n                RMSE_pp=100*np.sqrt(np.mean((y-p)**2)),\n                bias_pp=100*np.mean(p-y),\n                PI90_coverage=np.mean((y>=sub.lower90)&(y<=sub.upper90)),\n                PI90_width_pp=100*np.mean(sub.upper90-sub.lower90),\n                high_n=int(high.sum()),\n                high_bias_pp=100*np.mean((p-y)[high]) if high.any() else np.nan,\n                high_PI90_coverage=np.mean(\n                    (y[high]>=sub.lower90.to_numpy()[high])&\n                    (y[high]<=sub.upper90.to_numpy()[high])) if high.any() else np.nan,\n                ensemble_sd_pp=100*np.mean(sub.ensemble_sd))\ndef novelty_fit(tr,query):\n    # PCA nearest-neighbour support check: exploratory, not a validated AOA guarantee.\n    trX=tr[FEATURE_COLS].to_numpy(np.float32)\n    qX=query[FEATURE_COLS].to_numpy(np.float32)\n    scaler=StandardScaler().fit(trX)\n    z=scaler.transform(trX);q=scaler.transform(qX)\n    pca=PCA(n_components=min(10,z.shape[1],len(z)-1),random_state=SEED).fit(z)\n    z=pca.transform(z);q=pca.transform(q)\n    nn=NearestNeighbors(n_neighbors=2).fit(z)\n    training_distance=nn.kneighbors(z)[0][:,1]\n    test_distance=nn.kneighbors(q,n_neighbors=1)[0][:,0]\n    limit=float(np.quantile(training_distance,.95))\n    return test_distance,limit\nall_results=[];all_predictions=[];all_cal=[]\nfor repeat in range(REPEATS):\n    cache=VALID_DIR/f\"heldout_stratified_v3_{SCALE_M}m_rep{repeat}.parquet\"\n    calibration_meta=VALID_DIR/f\"calibration_stratified_v3_{SCALE_M}m_rep{repeat}.json\"\n    if cache.exists() and calibration_meta.exists() and not FORCE_VALIDATION:\n        test_out=pd.read_parquet(cache)\n        info=json.loads(calibration_meta.read_text())\n        print(\"CACHED VALIDATION\",repeat,flush=True)\n    else:\n        train,cal,test=split_spatial(df,repeat)\n        models,preds=fit_ensemble(train,[cal,test],SEED+repeat*193)\n        cal_mean,_=preds[0];test_mean,test_sd=preds[1]\n        calibration=forest_balanced_calibration(cal,cal_mean)\n        lower,upper=interval_bounds(test_mean,calibration)\n        dist,limit=novelty_fit(train,test)\n        test_out=test[[\"site\",\"group\",\"row\",\"col\"]].copy()\n        test_out[\"observed\"]=test.mean_probability.to_numpy(float)\n        test_out[\"predicted\"]=test_mean\n        test_out[\"lower90\"]=lower\n        test_out[\"upper90\"]=upper\n        test_out[\"ensemble_sd\"]=test_sd\n        test_out[\"novelty_distance\"]=dist\n        test_out[\"novel\"]=dist>limit\n        info=dict(repeat=repeat,n_train=len(train),n_cal=len(cal),n_test=len(test),\n                  n_cal_each_forest=calibration[\"n_per_forest\"],\n                  strategy=TRAIN_STRATEGY,training_cap=TRAIN_CAP_PER_FOREST,\n                  calibration=calibration,novelty_limit=limit)\n        test_out.to_parquet(cache,index=False)\n        calibration_meta.write_text(json.dumps(info,indent=2))\n        print(\"VALIDATED\",repeat,info,flush=True)\n    all_predictions.append(test_out)\n    all_cal.append(info)\n    for forest,chunk in test_out.groupby(\"site\"):\n        all_results.append(dict(repeat=repeat,forest=forest,**score(chunk)))\nmetrics=pd.DataFrame(all_results)\nmetrics.to_csv(VALID_DIR/f\"per_forest_audit_stratified_v3_{SCALE_M}m.csv\",index=False)\n# GLOBAL metrics ONLY; equal weight to forests and equal weight to spatial repetitions.\nnumerical=[\"MAE_pp\",\"RMSE_pp\",\"bias_pp\",\"PI90_coverage\",\"PI90_width_pp\",\n           \"high_PI90_coverage\",\"ensemble_sd_pp\"]\nsummary=metrics.groupby(\"repeat\")[numerical].mean(numeric_only=True)\n# R² needs a global pooled definition, not averaging per-forest R².\nfor rep,pred in enumerate(all_predictions):\n    y=pred.observed.to_numpy(float);p=pred.predicted.to_numpy(float)\n    forest=pred.site.to_numpy()\n    w=np.array([1/(len(FORESTS)*np.sum(forest==s)) for s in forest])\n    mu=np.dot(w,y);total=np.dot(w,(y-mu)**2)\n    summary.loc[rep,\"balanced_global_R2\"]=1-np.dot(w,(y-p)**2)/total\n    summary.loc[rep,\"n_test\"]=len(pred)\n    summary.loc[rep,\"n_test_high\"]=int(np.sum(y>=.4))\nsummary.to_csv(VALID_DIR/f\"global_metrics_by_repeat_stratified_v3_{SCALE_M}m.csv\")\nglobal_means=summary.mean(numeric_only=True).to_frame(\"mean\").join(\n    summary.std(numeric_only=True).to_frame(\"sd\"))\nglobal_means.to_csv(VALID_DIR/f\"global_metrics_stratified_v3_{SCALE_M}m.csv\")\nprint(\"GLOBAL EQUAL-FOREST SPATIAL VALIDATION (mean and SD across repeats)\")\ndisplay(global_means.round(3))\nprint(\"Calibration metadata:\",all_cal)\nprint(\"Spatial test predictions and per-forest audits saved; global metrics are primary.\")\n# Compare EXACT same held-out spatial cells with archived random-training baseline.\nbaseline_rows=[];matched_rows=[]\nfor rep,new in enumerate(all_predictions):\n    original=VALID_DIR/f\"heldout_asymmetric_v2_{SCALE_M}m_rep{rep}.parquet\"\n    if not original.exists():\n        print(\"BASELINE MISSING\",original,flush=True);continue\n    old=pd.read_parquet(original)\n    keys=[\"site\",\"group\",\"row\",\"col\"]\n    common=new.merge(old,on=keys,suffixes=(\"_new\",\"_baseline\"),validate=\"one_to_one\")\n    assert len(common)==len(new)==len(old),(\n        \"Different holdout cells; cannot claim paired comparison\",rep,len(common),len(new),len(old))\n    assert np.allclose(common.observed_new,common.observed_baseline,atol=1e-7)\n    for label in [\"new\",\"baseline\"]:\n        y=common[f\"observed_{label}\"].to_numpy(float)\n        p=common[f\"predicted_{label}\"].to_numpy(float)\n        forests=common.site.to_numpy()\n        weights=np.array([1/(len(FORESTS)*np.sum(forests==site)) for site in forests])\n        mu=np.dot(weights,y)\n        high=y>=.4\n        covered=(y>=common[f\"lower90_{label}\"].to_numpy(float))&(\n                 y<=common[f\"upper90_{label}\"].to_numpy(float))\n        baseline_rows.append(dict(repeat=rep,model=label,n_test=len(y),n_high=int(high.sum()),\n          balanced_R2=1-np.dot(weights,(y-p)**2)/np.dot(weights,(y-mu)**2),\n          balanced_MAE_pp=sum(100*np.mean(abs((y-p)[forests==site])) for site in FORESTS)/len(FORESTS),\n          high_MAE_pp=100*np.mean(abs((y-p)[high])) if high.any() else np.nan,\n          high_bias_pp=100*np.mean((p-y)[high]) if high.any() else np.nan,\n          high_PI90_coverage=float(np.mean(covered[high])) if high.any() else np.nan,\n          overall_PI90_coverage=sum(np.mean(covered[forests==site]) for site in FORESTS)/len(FORESTS)))\nif baseline_rows:\n    comparison=pd.DataFrame(baseline_rows)\n    comparison.to_csv(VALID_DIR/f\"paired_random_vs_stratified_{SCALE_M}m.csv\",index=False)\n    print(\"\\\\n=== PAIRED PERFORMANCE: same test cells, random vs high-cover stratified training ===\")\n    display(comparison.groupby(\"model\").mean(numeric_only=True).round(3))\nelse:\n    print(\"No archived v2 baseline found; new metrics saved independently.\")\n\n\n# === 07B — REFINE INTERVALS ONLY; freeze saved v3 point predictions ===\n# Fit the IDENTICAL stratified ensembles to reconstruct calibration predictions,\n# but always retain the archived v3 point predictions / ensemble SD on test rows.\n# Full calibration observations are used with equal total weight per forest.\ndef weighted_tail(values,weights,q):\n    order=np.argsort(values)\n    x=np.asarray(values,float)[order]\n    w=np.asarray(weights,float)[order]\n    return float(x[min(len(x)-1,np.searchsorted(np.cumsum(w)/w.sum(),q,side=\"left\"))])\ndef weighted_effective_n(w):\n    w=np.asarray(w,float)\n    return float((w.sum()**2)/np.sum(w**2))\ndef weighted_signed_calibration(cal,pred):\n    y=cal.mean_probability.to_numpy(float)\n    p=np.asarray(pred,float)\n    forest=cal.site.to_numpy()\n    weights=np.array([1/(len(FORESTS)*np.sum(forest==site)) for site in forest])\n    error=y-p\n    alpha=1-INTERVAL_LEVEL\n    global_limits=[weighted_tail(error,weights,alpha/2),\n                   weighted_tail(error,weights,1-alpha/2)]\n    segments=[]\n    # Continuously shrink bin-specific quantiles towards pooled limits,\n    # avoiding the prior hard switch at 25 equal-subsampled observations.\n    for lo,hi in [(0,.10),(.10,.25),(.25,1.000001)]:\n        mask=(p>=lo)&(p<hi)\n        count=int(mask.sum())\n        if count>=10:\n            local=[weighted_tail(error[mask],weights[mask],alpha/2),\n                   weighted_tail(error[mask],weights[mask],1-alpha/2)]\n            eff_n=weighted_effective_n(weights[mask])\n            strength=min(.75,eff_n/(eff_n+40))\n        else:\n            local=global_limits\n            eff_n=weighted_effective_n(weights[mask]) if count else 0\n            strength=0.\n        bounds=((1-strength)*np.asarray(global_limits)+\n                strength*np.asarray(local)).tolist()\n        segments.append(dict(min_pred=lo,max_pred=hi,n_cal=count,\n                weighted_effective_n=round(eff_n,2),local_weight=strength,\n                limits=bounds,raw_local_limits=local))\n    return dict(n_cal=len(y),n_cal_by_forest={site:int((forest==site).sum())\n                for site in FORESTS},global_limits=global_limits,bins=segments)\nrefined_rows=[];refined_predictions=[];refined_cal=[]\nfor repeat in range(REPEATS):\n    old=all_predictions[repeat].copy()\n    save=VALID_DIR/f\"heldout_weightedcal_v4_{SCALE_M}m_rep{repeat}.parquet\"\n    meta=VALID_DIR/f\"calibration_weightedcal_v4_{SCALE_M}m_rep{repeat}.json\"\n    if save.exists() and meta.exists() and not FORCE_VALIDATION:\n        refreshed=pd.read_parquet(save)\n        info=json.loads(meta.read_text())\n        print(\"CACHED WEIGHTED CALIBRATION\",repeat,flush=True)\n    else:\n        train,cal,test=split_spatial(df,repeat)\n        _,predictions=fit_ensemble(train,[cal,test],SEED+repeat*193)\n        cal_prediction,_=predictions[0]\n        test_prediction,test_sd=predictions[1]\n        keys=[\"site\",\"group\",\"row\",\"col\"]\n        check=test[keys].copy()\n        check[\"predicted\"]=test_prediction\n        check[\"observed\"]=test.mean_probability.to_numpy(float)\n        matched=old.merge(check,on=keys,suffixes=(\"_old\",\"_recomputed\"),\n                          validate=\"one_to_one\",sort=False)\n        assert len(matched)==len(old)==len(test),\"Geographic split has changed\"\n        assert np.allclose(matched.predicted_old,matched.predicted_recomputed,atol=2e-6),(\n            \"Point predictions have changed; calibration-only comparison invalid\")\n        assert np.allclose(matched.observed_old,matched.observed_recomputed,atol=1e-7)\n        calibration=weighted_signed_calibration(cal,cal_prediction)\n        refreshed=old.copy()\n        refreshed[\"lower90\"],refreshed[\"upper90\"]=interval_bounds(\n            refreshed.predicted.to_numpy(float),calibration)\n        info=dict(repeat=repeat,n_train=len(train),n_cal=len(cal),\n                  n_test=len(test),calibration=calibration,\n                  source_point_predictions=\"heldout_stratified_v3\",model_unchanged=True)\n        refreshed.to_parquet(save,index=False)\n        meta.write_text(json.dumps(info,indent=2))\n        print(\"WEIGHTED CALIBRATION\",repeat,\"n_cal\",len(cal),\n              \"weighted effective support\",\n              [round(i[\"weighted_effective_n\"],1) for i in calibration[\"bins\"]],\n              flush=True)\n    assert len(refreshed)==len(old) and np.allclose(\n        refreshed.predicted,old.predicted,atol=1e-7)\n    refined_predictions.append(refreshed)\n    refined_cal.append(info)\n    for model,sub in [(\"previous_v3\",old),(\"weighted_v4\",refreshed)]:\n        y=sub.observed.to_numpy(float);p=sub.predicted.to_numpy(float)\n        forest=sub.site.to_numpy()\n        weights=np.array([1/(len(FORESTS)*np.sum(forest==site)) for site in forest])\n        covered=(y>=sub.lower90.to_numpy(float))&(y<=sub.upper90.to_numpy(float))\n        high=y>=.4\n        row=dict(repeat=repeat,model=model,n_test=len(sub),n_high=int(high.sum()),\n            balanced_global_coverage=sum(np.mean(covered[forest==site])\n                              for site in FORESTS)/len(FORESTS),\n            high_cover_coverage=np.mean(covered[high]) if high.any() else np.nan,\n            interval_width_pp=100*sum(np.mean(\n                (sub.upper90.to_numpy(float)-sub.lower90.to_numpy(float))[forest==site])\n                for site in FORESTS)/len(FORESTS),\n            balanced_MAE_pp=100*sum(np.mean(abs((y-p)[forest==site]))\n                for site in FORESTS)/len(FORESTS),\n            high_bias_pp=100*np.mean((p-y)[high]) if high.any() else np.nan,\n            novelty_coverage_inside=np.mean(covered[~sub.novel.to_numpy(bool)])\n                if (~sub.novel.to_numpy(bool)).any() else np.nan,\n            novelty_coverage_outside=np.mean(covered[sub.novel.to_numpy(bool)])\n                if sub.novel.to_numpy(bool).any() else np.nan,\n            novelty_MAE_inside_pp=100*np.mean(abs((y-p)[~sub.novel.to_numpy(bool)]))\n                if (~sub.novel.to_numpy(bool)).any() else np.nan,\n            novelty_MAE_outside_pp=100*np.mean(abs((y-p)[sub.novel.to_numpy(bool)]))\n                if sub.novel.to_numpy(bool).any() else np.nan)\n        refined_rows.append(row)\nrefined_compare=pd.DataFrame(refined_rows)\nrefined_compare.to_csv(VALID_DIR/f\"paired_interval_calibration_v3_v4_{SCALE_M}m.csv\",index=False)\nprint(\"\\n=== IDENTICAL PREDICTIONS: OLD VS ALL-CALIBRATION WEIGHTED INTERVALS ===\",flush=True)\ndisplay(refined_compare.groupby(\"model\").mean(numeric_only=True).round(3))\nprint(\"\\n=== PREDICTION INTERVAL COVERAGE BY OBSERVED ABUNDANCE (audit only) ===\")\nabundance_rows=[]\nfor repeat,old in enumerate(all_predictions):\n    for model,sub in [(\"previous_v3\",old),(\"weighted_v4\",refined_predictions[repeat])]:\n        y=sub.observed.to_numpy(float)\n        in_interval=(y>=sub.lower90.to_numpy(float))&(\n                     y<=sub.upper90.to_numpy(float))\n        for label,mask in [(\"<20%\",y<.20),(\"20-40%\",(y>=.20)&(y<.40)),\n                           (\">=40%\",y>=.40)]:\n            if mask.any():\n                abundance_rows.append(dict(repeat=repeat,model=model,\n                    observed_class=label,n=int(mask.sum()),\n                    coverage=float(np.mean(in_interval[mask]))))\nabundance_comp=pd.DataFrame(abundance_rows)\nabundance_comp.to_csv(VALID_DIR/f\"paired_abundance_coverage_v3_v4_{SCALE_M}m.csv\",index=False)\ndisplay(abundance_comp.groupby([\"model\",\"observed_class\"]).apply(\n    lambda g:pd.Series(dict(n_total=g.n.sum(),\n         pooled_coverage=np.average(g.coverage,weights=g.n))),include_groups=False).round(3))\n# Export stage consumes the latest weighted calibration only, after coverage review.\nmapping_cal=refined_cal\nprint(\"Refinement done. Hold Block 09 until coverage is assessed.\",flush=True)\n"
print('Loaded verified 07 code:',len(BASELINE_07.splitlines()),'lines')

## 03 — Run per-resolution spatially held-out experiments

Set `DRY_RUN=False` in Section 01 only after reviewing the input inventory. Missing per-scale caches are reported rather than synthesized; this avoids accidental mixing of satellite and UAV grids. **Each scenario is independent and resumable** using its own cached heldout outputs.

In [ ]:
# 03 — Run reusable balanced-forest spatial validation at 5 resolutions
# Copy of the VERIFIED old 06 + 07 implementation, with only scale/input/output paths parameterized.
# Do NOT set DRY_RUN=False before inspecting the inventory in Section 01.
if DRY_RUN:
    print("DRY RUN: no fitting. Available scenarios:",[(int(r.scale_m),r.variant)
      for r in inventory.itertuples() if r.exists and getattr(r,"valid",False)])
else:
    finished=[];missing=[]
    for scale in SCALES:
        for variant in (["RAW"] if not RUN_BRDF else ["RAW","BRDF"]):
            if variant=="RAW" and not RUN_RAW:continue
            entry=inventory[(inventory.scale_m==scale)&(inventory.variant==variant)].iloc[0]
            if not entry.exists or not bool(entry.get("valid",False)):
                print("SKIP: missing or incompatible input",scale,variant,entry.path,flush=True)
                missing.append((scale,variant))
                continue
            EXPERIMENT_SCALE_M=int(scale)
            EXPERIMENT_SAMPLE_PATH=Path(entry.path)
            EXPERIMENT_VALID_DIR=FINAL/f"{scale}m"/variant/"validation"
            print("BEGIN",scale,variant,"cells",entry.n_cells,flush=True)
            # Run same working fit/splits/calibration code. Unchanged logic, fresh scenario namespace.
            exec(compile(BASELINE_06,"NB04_verified_06.py","exec"),globals())
            exec(compile(BASELINE_07,"NB04_verified_07.py","exec"),globals())
            finished.append((scale,variant))
            print("FINISHED",scale,variant,flush=True)
    print("Complete scenarios:",finished,"missing:",missing)


## 04 — Results: equal-forest statistics, site-specific performance, reproducible held-out predictions

In [ ]:
# 04 — Collect held-out predictions, compare scales and paired BRDF effects
# Safe to rerun; only reads scenario-specific Section 07 files.
from pathlib import Path
from IPython.display import display
from sklearn.metrics import r2_score
import numpy as np,pandas as pd
records=[];predictions=[]
for scale in SCALES:
    for variant in ("RAW","BRDF"):
        directory=FINAL/f"{scale}m"/variant/"validation"
        paths=[directory/f"heldout_weightedcal_v4_{scale}m_rep{k}.parquet" for k in range(3)]
        if not all(p.exists() for p in paths):
            print("NOT YET VALIDATED",scale,variant,
                  [str(p.name) for p in paths if not p.exists()])
            continue
        for repeat,p in enumerate(paths):
            d=pd.read_parquet(p)
            assert {"site","group","row","col","observed","predicted"}.issubset(d.columns),(p,d.columns)
            d["resolution_m"]=scale;d["variant"]=variant;d["repeat"]=repeat
            predictions.append(d)
            for site,g in d.groupby("site"):
                y=g.observed.to_numpy(float);pred=g.predicted.to_numpy(float)
                high=y>=.4
                coverage=np.nan
                if {"lower90","upper90"}.issubset(g):
                    coverage=float(np.mean((y>=g.lower90)&(y<=g.upper90)))
                records.append(dict(scale_m=scale,variant=variant,repeat=repeat,site=site,
                    n=len(g),n_high=int(high.sum()),
                    MAE_pp=100*np.mean(np.abs(pred-y)),
                    RMSE_pp=100*np.sqrt(np.mean((pred-y)**2)),
                    bias_pp=100*np.mean(pred-y),
                    R2=np.nan if np.var(y)<1e-12 else r2_score(y,pred),
                    high_bias_pp=100*np.mean((pred-y)[high]) if high.any() else np.nan,
                    high_MAE_pp=100*np.mean(np.abs((pred-y)[high])) if high.any() else np.nan,
                    PI90_coverage=coverage))
if not records:
    print("No completed scenario validation results yet; run Section 03 with DRY_RUN=False.")
else:
    per=pd.DataFrame(records)
    per.to_csv(FINAL/"site_metrics_all_resolutions.csv",index=False)
    global_=per.groupby(["scale_m","variant","repeat"],as_index=False).agg(
        MAE_pp=("MAE_pp","mean"),RMSE_pp=("RMSE_pp","mean"),
        bias_pp=("bias_pp","mean"),R2_mean_site=("R2","mean"),
        high_bias_pp=("high_bias_pp","mean"),
        high_MAE_pp=("high_MAE_pp","mean"),
        PI90_coverage=("PI90_coverage","mean"))
    summary=global_.groupby(["scale_m","variant"]).agg(["mean","std"])
    summary.to_csv(FINAL/"global_equal_forest_summary.csv")
    display(summary.round(3))
    byforest=per.groupby(["scale_m","variant","site"]).mean(numeric_only=True)
    display(byforest.round(3))
    pred=pd.concat(predictions,ignore_index=True)
    pred.to_parquet(FINAL/"all_spatial_holdout_predictions.parquet",index=False)
    paired=[]
    for (scale,rep),g in pred.groupby(["resolution_m","repeat"]):
        a=g[g.variant=="RAW"];b=g[g.variant=="BRDF"]
        if a.empty or b.empty:continue
        keys=["site","group","row","col"]
        m=a.merge(b,on=keys,suffixes=("_raw","_brdf"),validate="one_to_one")
        assert len(m)==len(a)==len(b),(scale,rep,"Holdout cell mismatch")
        assert np.allclose(m.observed_raw,m.observed_brdf,atol=1e-7)
        for site,z in m.groupby("site"):
            raw=100*np.abs(z.predicted_raw-z.observed_raw).mean()
            brdf=100*np.abs(z.predicted_brdf-z.observed_brdf).mean()
            paired.append(dict(scale_m=scale,repeat=rep,site=site,n=len(z),
                raw_MAE_pp=raw,brdf_MAE_pp=brdf,delta_MAE_BRDF_minus_RAW_pp=brdf-raw))
    if paired:
        diff=pd.DataFrame(paired)
        diff.to_csv(FINAL/"paired_BRDF_vs_RAW_same_holdout_cells.csv",index=False)
        display(diff.groupby(["scale_m","site"]).delta_MAE_BRDF_minus_RAW_pp.agg(["mean","std"]).round(3))
    else:
        print("BRDF pairs unavailable until original V3 correction is recovered and corrected inputs generated.")


## 05 — BRDF correction provenance / preparation (required before paired BRDF model fitting)

**Current status: the existing V3 BRDF output folders are identified, but their actual raster content, metadata and geographic alignment have not yet been verified.** Do not improvise a different normalization and call it a paired V3 replication.

Required step: locate V3's actual Earth Engine/Colab BRDF routine and freeze: Sentinel-2 source, sun/view angles, model coefficients/kernels, target geometry, cloud masking and date windows; apply to identical 10/20 m scenes **before** calculating quarterly median and spatial aggregate. Save one verified corrected feature parquet per resolution `roi_cells_{scale}m_BRDF.parquet` in the samples directory with the **same** `EMB_ANNUAL__` and `S2_QUARTERLY_3YR__` feature column names and raw-table rows/targets. Preserve a processing metadata JSON alongside it.

Crucially, BRDF correction changes **only the satellite reflectance features**; never the UAV fractions, annual embeddings, spatial sampling, heldout allocation or per-site balancing. Check corrected-vs-raw reflectance differences are nonzero and plausible and ensure all 44 seasonal bands, including count features, remain aligned.

The existing 100 m model and its maps must not be overwritten by these experiments.